In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from pathlib import Path

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough, chain
from langchain_core.prompts import ChatPromptTemplate
from util import VectorStoreRetrieverWithTextSplitter
from langchain.storage import InMemoryByteStore
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.output_parsers import StrOutputParser
from meeplemate.llm_models import (
    load_tgi_chat_model,
    load_vllm_chat_model,
    load_tokenizer,
    load_jina_embedding_model,
    sentence_transformer_to_hf_embeddings
)
from meeplemate.pdf import parse_pdf

/workspaces/mistral-rag-game-rules/code/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspaces/mistral-rag-game-rules/code/.venv/lib/python3.10/site-packages/pydantic/_internal/_fields.py:151: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [3]:
# Data
model_name='teknium/OpenHermes-2.5-Mistral-7B'
data_path = Path("./munchkin_rules/")
tgi_url = "http://tgi:80"

In [4]:
# nltk is used for PDF processing. Here we ensure anything it downloads goes to
# the cache folder, so it doesn't have to download again
nltk_data_path = Path("~/.cache/nltk_data").expanduser()
nltk_data_path.mkdir(parents=True, exist_ok=True)
os.environ["NLTK_DATA"] = str(nltk_data_path)

In [5]:
!sudo apt-get update -y
!sudo apt-get install -y poppler-utils tesseract-ocr

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 83 not upgraded.


In [6]:
tokenizer = load_tokenizer(model_name)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [7]:
def load_docs():
    rule_docs = []
    for filename in data_path.glob("*.pdf"):
        print(f"Processing {filename}")
        rule_docs.extend(parse_pdf(filename))
    return rule_docs

In [8]:
rule_docs = load_docs()

Processing munchkin_rules/munchkin_rules-1.pdf
Processing munchkin_rules/puppies-rules.pdf
Processing munchkin_rules/princesses_rules.pdf
Processing munchkin_rules/munch_4_rules_20thp.pdf


In [11]:
chat_model = load_tgi_chat_model(
    tokenizer=tokenizer,
    inference_server_url=tgi_url,
    max_new_tokens=512,
    timeout=900,
    do_sample=False,
    temperature=0.01,
)

In [12]:
from meeplemate.retrievers import (
    build_retriever_for_documents,
    build_retriever_with_hypothetical_questions_for_documents
)
from meeplemate.vectorstores import build_vectorstore_faiss

jina_embedding_model = load_jina_embedding_model()
hf_embedding_model = sentence_transformer_to_hf_embeddings(jina_embedding_model, normalize_embeddings=True)
db = build_vectorstore_faiss(hf_embedding_model)
parent_document_retriever = build_retriever_for_documents(
    tokenizer,
    db,
    rule_docs,
    k=20
)

hypothetical_queries_document_retriever = build_retriever_with_hypothetical_questions_for_documents(
    tokenizer,
    db,
    chat_model,
    rule_docs,
    k=20
)

In [13]:
for doc in parent_document_retriever.invoke("When can I discard a Race card?"):
    print(doc.page_content)
    print()

CHARACTER STATS
Each character is basically a collection of weapons, armor, and magic items, with three stats: Level, Race, and Class. For instance, you might describe your character as a “Level 8 Elf Wizard with Boots of Butt-Kicking, a Staff of Napalm, and the Kneepads of Allure.” Level: This is a measure of how generally buff and studly you are. When the rules or cards refer to your Level, capitalized, they mean this number. You gain a level when you kill a monster, or when a card says that you do. You can also sell Items to buy levels (see Items). You lose a level when a card says you do. Your Level can never go below 1. However, your combat strength can be negative, if you get hit by a Curse or suffer some other kind of penalty.
Class: Characters may be Warriors, Wizards, Thieves, or Clerics. If you have no Class card in front of you, you have no class. Yeah, I know, we did that one already.
Each Class has special abilities, shown on the cards. You gain the abilities of a Class th

In [15]:
hypothetical_queries_document_retriever.search_kwargs = {"k": 20, "fetch_k": 30}

for doc in hypothetical_queries_document_retriever.invoke("When can I discard a Race card?"):
    print(doc.page_content)
    print()

CHARACTER STATS
Each character is basically a collection of weapons, armor, and magic items, with three stats: Level, Race, and Class. For instance, you might describe your character as a “Level 8 Elf Wizard with Boots of Butt-Kicking, a Staff of Napalm, and the Kneepads of Allure.” Level: This is a measure of how generally buff and studly you are. When the rules or cards refer to your Level, capitalized, they mean this number. You gain a level when you kill a monster, or when a card says that you do. You can also sell Items to buy levels (see Items). You lose a level when a card says you do. Your Level can never go below 1. However, your combat strength can be negative, if you get hit by a Curse or suffer some other kind of penalty.
Class: Characters may be Warriors, Wizards, Thieves, or Clerics. If you have no Class card in front of you, you have no class. Yeah, I know, we did that one already.
Each Class has special abilities, shown on the cards. You gain the abilities of a Class th

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

cot_prompt_template = """\
Answer the following board game question based on the given rules from the \
rulebook. Provide your step-by-step reasoning first followed by the answer. \
Each step should be a separate bullet point. Remember rules found in a board \
game rulebook generally hold unless there is an explicit exception.

> Context:
>>>
{context}
>>>
> Question: {question}
> Answer: Let's think step by step. \
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("human", cot_prompt_template),
    ],
)

context_when_take_actions = """\
CHARACTER STATS
Each character is basically a collection of weapons, armor, and magic items, with three stats: Level, Race, and Class. For instance, you might describe your character as a “Level 8 Elf Wizard with Boots of Butt-Kicking, a Staff of Napalm, and the Kneepads of Allure.” Level: This is a measure of how generally buff and studly you are. When the rules or cards refer to your Level, capitalized, they mean this number. You gain a level when you kill a monster, or when a card says that you do. You can also sell Items to buy levels (see Items). You lose a level when a card says you do. Your Level can never go below 1. However, your combat strength can be negative, if you get hit by a Curse or suffer some other kind of penalty.
Class: Characters may be Warriors, Wizards, Thieves, or Clerics. If you have no Class card in front of you, you have no class. Yeah, I know, we did that one already.
Each Class has special abilities, shown on the cards. You gain the abilities of a Class the moment you play its card in front of you, and lose them as soon as you discard that card. Some Class abilities are powered by discards. You may discard any card, in play or in your hand, to power a special ability.
See the Class cards for when abilities can be used. Note that a Thief cannot steal while he or the target is fighting - and as soon as a monster is revealed, the fight is on!
You can discard a Class card at any time, even in combat: “I don't wanna be a wizard anymore.” When you discard a Class card, you become classless until you play another Class card.
You may not belong to more than one class at once unless you play the Super Munchkin card.
Race: Characters may be Humans, Elves, Dwarves, or Halflings. If you have no Race card in front of you, you are human.

Race: Characters may be Humans, Elves, Dwarves, or Halflings. If you have no Race card in front of you, you are human.
Humans have no special abilities. The rules for Classes, above, also apply to Races.
You may not belong to more than one race at once unless you play the Half-Breed card.
"""

question = "When can I discard a Race card?"

In [20]:
cot_chain = (prompt | chat_model | StrOutputParser())

print(
    cot_chain.invoke({"context": context_when_take_actions, "question": question})
)


Step 1: The rules mention that you can discard a Class card at any time, even in combat.
Step 2: The rules also mention that you may not belong to more than one class at once unless you play the Super Munchkin card.
Step 3: The rules state that "Race: Characters may be Humans, Elves, Dwarves, or Halflings. If you have no Race card in front of you, you are human."
Step 4: The rules also mention that "You may not belong to more than one race at once unless you play the Half-Breed card."
Step 5: There is no explicit rule about discarding a Race card.

Based on the given rules, there is no explicit rule about when you can discard a Race card. However, since the rules allow discarding a Class card at any time, it is reasonable to assume that you can discard a Race card at any time as well, as long as you follow the rules for belonging to more than one race.
